# Hybrid MPI+OpenMP Programming

Clusters are built from nodes with many cores that share memory.
So far, we have started one MPI process per core and let them communicate by messages — even within a node.
**Hybrid programming** combines MPI between processes with a shared-memory model within a process — here OpenMP threads.

Is that a good idea? It depends: hybrid programming can save memory, ease load balancing and enable communication overlap, but it adds a second level of parallelism with its own overheads and pitfalls.
This notebook gives you the tools to decide — and to do it right.

**After working through this notebook you will be able to**

* write simple OpenMP programs: parallel regions, worksharing loops, reductions,
* initialize MPI for threads with `MPI_Init_thread()` and choose the right thread support level,
* use the typical *funneled* pattern, and let several threads communicate at the same time,
* let threads contribute the parts of one message with partitioned communication (MPI 4.0),
* avoid the pitfalls of collectives called from several threads,
* start hybrid programs with the right number of processes and threads, pin them to cores, and place data in the right ccNUMA domain,
* recognize and fix load imbalance with OpenMP loop schedules,
* weigh the pros and cons of hybrid programming for a given code — including the alternatives that current MPI libraries offer.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The cells are compiled and run with the `%%mpi` and `%%c` cell magics introduced in [MPI in a Nutshell](mpi-in-a-nutshell.ipynb).
This notebook adds OpenMP:

| Option | Meaning |
|---|---|
| `-m` | enable OpenMP: adds the compiler flags and `#include <omp.h>`; `%%mpi` then initializes MPI with `MPI_Init_thread(…, MPI_THREAD_FUNNELED, &provided)` |
| `-e OMP_NUM_THREADS=N` | run with `N` threads per process |
| `-n N` | start `N` MPI processes (default: 4) |
| `-r` | the cell is a complete program (no wrapping) |
| `-t` | add timing code |
| `-O VAR` | store the program output in the Python variable `VAR` |
| `-L "…"` | replace the flags passed to `mpirun`, e.g. for pinning (see the section on pinning) |

The OpenMP flags are detected automatically.
On macOS, Apple's compiler needs the OpenMP runtime from Homebrew: `brew install libomp`.
With Open MPI, `-m` also passes `--bind-to none` to `mpirun`, so that the threads of a process can use more than one core (see the section on pinning).

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

## Why Hybrid?

* HPC clusters consist of large shared-memory nodes — using a shared-memory programming model within a node seems natural.
  Options are POSIX threads, OpenMP, or C++ models such as oneAPI/SYCL.
* GPU-accelerated clusters need a dedicated model for the GPUs anyway: CUDA (see GPU Programming with CUDA), HIP/ROCm, OpenMP or OpenACC offloading, SYCL.
  They typically run one MPI process per GPU — and the many remaining CPU cores of a node can only be used by threads.
* The reality: MPI knows little about the other programming model — the thread support level is all it learns about the threads.
  Start-up and process pinning are not standardized, and **MPI+X comes with considerable complexity**.

This notebook uses OpenMP, the most common partner of MPI on CPUs.

## OpenMP Primer

[OpenMP](https://www.openmp.org/specifications/) is a set of compiler directives (`#pragma omp …` in C/C++, `!$omp …` in Fortran), runtime functions and environment variables for thread parallelism.
It needs compiler support (`-fopenmp` and friends); the current version is OpenMP 6.0, which covers parallel regions, synchronization, tasking, SIMD, loop transformations and GPU offloading.

### Parallel Regions: Fork and Join

![Fork-join model: the initial thread runs serially; at a parallel region it forks a team of threads, which join again at the end of the region after an implicit barrier, and serial execution continues](img/fork-join.svg)

* The **initial thread** executes the serial parts.
* At a **parallel region**, it forks a **team of threads** — it becomes thread 0 of the team — which all execute the region.
* At the end of the region, the threads wait in an implicit **barrier** and join; serial execution continues.
* The number of threads is set by the environment variable `OMP_NUM_THREADS`; without it, the default is implementation-defined — usually one thread per core.

> 🤔 **Predict:** In which order do the threads print?

In [ ]:
%%c -m -e OMP_NUM_THREADS=4 -o code/hybrid-hello-omp.c
printf("serial part: %d thread\n", omp_get_num_threads());

#pragma omp parallel
{
    int id = omp_get_thread_num();          // declared in the region: private to each thread
    printf("hello from thread %d of %d\n", id, omp_get_num_threads());
}

printf("serial part again\n");

As with MPI processes, the output order of threads is not defined.
Variables declared inside a parallel region are **private** to each thread; variables declared outside are **shared** by all threads.

### Worksharing Loops and Reductions

```c
#pragma omp parallel for [clauses]
for (int i = 0; i < n; ++i) { … }
```

`#pragma omp for` inside a parallel region distributes the iterations of the following loop among the threads; `parallel for` combines both.
The loop counter is private, there is an implicit barrier at the end, and **the iterations must be independent**.
A `reduction(+:sum)` clause gives every thread a private copy of `sum` and adds them up at the end — the shared-memory counterpart of `MPI_Reduce()`.

Here is the computation of $\pi$ from [MPI in a Nutshell](mpi-in-a-nutshell.ipynb), parallelized with OpenMP:

In [ ]:
%%c -m -e OMP_NUM_THREADS=4 -t -o code/hybrid-pi-omp.c
const int64_t N = 1000000000;
const double h = 1.0 / N;

double sum = 0.0;
#pragma omp parallel for reduction(+:sum)
for (int64_t i = 0; i < N; ++i) {
    double x = (i + 0.5) * h;
    sum += 4.0 / (1.0 + x * x);
}

printf("pi = %.15f with %d threads\n", sum * h, omp_get_max_threads());

> 🤔 **Predict:** What happens without the `reduction(+:sum)` clause? Try it, and run the cell a few times.
> Then compare the runtime with `OMP_NUM_THREADS=1`, `2` and `4`.

### Thread Placement: `OMP_PLACES` and `OMP_PROC_BIND`

By default, the operating system may move threads from core to core at any time.
For reliable performance — and, with several processes per node, to use every core exactly once — threads should be **pinned**: bound to fixed cores.
OpenMP controls this with two environment variables.

**`OMP_PLACES`** defines the **places**: the units of hardware a thread can be bound to.
A thread bound to a place may run on any CPU of that place, but nowhere else.

| `OMP_PLACES` | one place per |
|---|---|
| `threads` | hardware thread (with SMT, a core has several) |
| `cores` | physical core, with all its hardware threads |
| `ll_caches` | group of cores sharing a last-level cache |
| `numa_domains` | ccNUMA domain |
| `sockets` | socket |
| `"{0},{1},{4},{5}"` | explicit list of places, given by CPU (hardware thread) numbers |

Here are the places on a node with 2 sockets × 4 cores and 2 hardware threads per core.
The numbers are the CPU numbers the operating system uses; core $c$ has the CPUs $c$ and $c + 8$, a common numbering on Linux:

![The places of four OMP_PLACES settings on a node with 2 sockets of 4 cores with 2 hardware threads each: threads gives 16 places of one CPU each; cores gives 8 places of the two CPUs of a core; sockets gives 2 places of 8 CPUs each; the explicit list {0},{1},{4},{5} gives 4 places of the CPUs 0, 1, 4 and 5](img/omp-places.svg)

**`OMP_PROC_BIND`** decides how the threads of a team are distributed over the places:

| `OMP_PROC_BIND` | thread placement |
|---|---|
| `close` | on consecutive places, starting at the place of the primary thread |
| `spread` | evenly over all places, as far apart as possible |
| `primary` (formerly `master`) | all threads on the place of the primary thread |
| `true` / `false` | bind with an implementation-defined policy / don't bind at all |

On the same node with `OMP_PLACES=cores` and `OMP_NUM_THREADS=4`:

* `OMP_PROC_BIND=close` puts the threads on cores 0, 1, 2, 3 — all on socket 0, sharing its cache and memory bandwidth,
* `OMP_PROC_BIND=spread` puts them on cores 0, 2, 4, 6 — on both sockets, with twice the memory bandwidth.

![Thread placement with OMP_PLACES=cores: with close, threads T0 to T3 run on cores 0 to 3 of socket 0; with spread, they run on cores 0, 2, 4 and 6 of both sockets; with 2 processes of 2 threads each pinned to one socket, spread places the threads of process 0 on cores 0 and 2 and those of process 1 on cores 4 and 6](img/omp-proc-bind.svg)

Set both variables; otherwise the binding is implementation-defined, often none at all.
**Important for hybrid programs:** the places are formed only from the CPUs the *process* is allowed to run on.
The MPI launcher pins each process to a set of cores, and `OMP_PLACES`/`OMP_PROC_BIND` distribute its threads *within* that set — in the last row above, each process sees only the 4 cores of its socket, and `spread` uses every other one of them.
See the section on starting and pinning for the launcher options.
With `OMP_DISPLAY_AFFINITY=true`, the OpenMP runtime reports where each thread is bound.

## MPI and Threads

### Thread Support Levels

A program with threads must tell MPI what the threads will do:

```c
int MPI_Init_thread(int *argc, char ***argv, int required, int *provided);
```

replaces `MPI_Init()`.
`required` is one of four levels, in increasing order:

| Level | Meaning |
|---|---|
| `MPI_THREAD_SINGLE` | only one thread will execute |
| `MPI_THREAD_FUNNELED` | the process may be multi-threaded, but only the **main thread** (the one that called `MPI_Init_thread()`) calls MPI |
| `MPI_THREAD_SERIALIZED` | any thread may call MPI, but **only one at a time** |
| `MPI_THREAD_MULTIPLE` | any thread may call MPI at any time, without restrictions |

![The three thread support levels with threads: with FUNNELED, MPI is called only by the main thread, outside parallel regions or by the main thread inside; with SERIALIZED, different threads call MPI, but never at the same time; with MULTIPLE, all threads may call MPI concurrently](img/thread-levels.svg)

* MPI returns the level it supports in `provided` — this may be lower (or higher) than `required`.
  **Check it**, and abort if it is lower than what your program needs.
* With `MPI_Init()`, the level is `MPI_THREAD_SINGLE` in principle — strictly, not even threads that never call MPI are allowed.
  Any OpenMP program needs at least `MPI_THREAD_FUNNELED`; that's why `%%mpi -m` initializes MPI with it.
  (The lecture slides use `MPI_THREAD_MULTIPLE` as the check for "any threading" — that's only needed if several threads call MPI concurrently.)
* Higher levels may cost performance: with `MPI_THREAD_MULTIPLE`, the library has to protect its internal state, and a single lock would serialize the communication of all threads.
  Current libraries do much better if the threads communicate on **different communicators** — see the section on collectives from several threads below.
* With the **Sessions** model of MPI 4.0, a library can initialize MPI on its own with `MPI_Session_init()` and request its level with the info key `thread_level` (as a string, e.g. `"MPI_THREAD_MULTIPLE"`).
  Whether different sessions of a process can really have different levels is up to the implementation — MPICH, e.g., keeps the first level for the whole process.

`MPI_Init_thread()` can be called only once per process, so the following complete program (`-r`) requests the highest level, `MPI_THREAD_MULTIPLE`, and reports what it gets.
`MPI_Query_thread()` returns the provided level at any later time, and `MPI_Is_thread_main()` tells a thread whether it is the main thread:

In [ ]:
%%mpi -r -m -n 2 -e OMP_NUM_THREADS=2 -o code/hybrid-init-thread.c
#include <stdio.h>
#include <mpi.h>
#include <omp.h>

const char *level_name(int level) {
    return level == MPI_THREAD_SINGLE     ? "MPI_THREAD_SINGLE"
         : level == MPI_THREAD_FUNNELED   ? "MPI_THREAD_FUNNELED"
         : level == MPI_THREAD_SERIALIZED ? "MPI_THREAD_SERIALIZED"
         :                                  "MPI_THREAD_MULTIPLE";
}

int main(int argc, char **argv) {
    int provided, rank;

    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    int queried;
    MPI_Query_thread(&queried);                     // the provided level, available at any time
    if (rank == 0)
        printf("required MPI_THREAD_MULTIPLE, provided %s (MPI_Query_thread: %s)\n", level_name(provided), level_name(queried));

    #pragma omp parallel
    {
        int is_main;
        MPI_Is_thread_main(&is_main);               // allowed from any thread
        printf("rank %d, thread %d: main thread? %s\n", rank, omp_get_thread_num(), is_main ? "yes" : "no");
    }

    MPI_Finalize();
    return 0;
}

### The Funneled Pattern

The most common — and most portable — hybrid pattern: the threads share the computation, and **only the main thread communicates**, either outside of parallel regions or inside them in a `masked` (formerly `master`) block.
Since `masked` has **no implied barrier**, the other threads must wait explicitly before they use the result.

The following cell normalizes a distributed vector three times.
In every iteration, all threads compute the local part of the squared norm, the main thread combines it with `MPI_Allreduce()`, and all threads scale their part of the vector.

In [ ]:
%%mpi -m -n 2 -e OMP_NUM_THREADS=3 -i math.h -f lm -o code/hybrid-funneled.c
const int n = 1000000;                          // local vector length
double *x = malloc(n * sizeof(double));
for (int i = 0; i < n; ++i)
    x[i] = rank + 1.0;

double local = 0.0, global = 0.0;               // shared by all threads

#pragma omp parallel
{
    for (int iter = 0; iter < 3; ++iter) {
        #pragma omp single
        local = 0.0;                            // implicit barrier at the end of single

        // all threads: the local part of the squared norm
        #pragma omp for reduction(+:local)
        for (int i = 0; i < n; ++i)
            local += x[i] * x[i];               // implicit barrier at the end of the loop

        // only the main thread calls MPI (MPI_THREAD_FUNNELED)
        #pragma omp masked
        {
            MPI_Allreduce(&local, &global, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
            if (rank == 0)
                printf("iteration %d: norm = %.6f\n", iter, sqrt(global));
        }
        #pragma omp barrier                     // masked has no implied barrier: wait for the result

        // all threads: normalize
        #pragma omp for
        for (int i = 0; i < n; ++i)
            x[i] /= sqrt(global);
    }
}

free(x);

> 🤔 **Predict:** What can go wrong if you remove the `#pragma omp barrier`?

`masked` replaced `master` in OpenMP 5.1 (supported since GCC 12 and LLVM/clang 14); `master` still works, but is deprecated — use it only with older compilers.

### Multiple Threads Communicating

With `MPI_THREAD_MULTIPLE`, every thread may communicate on its own.
Then the usual matching rules apply to the messages of *all* threads of a process — MPI does not know about threads.
To keep the conversations of different threads apart, use **different tags** (or communicators).

Here, every thread exchanges a message with the thread of the same number on the other process, with the thread number as tag.

> 🤔 **Predict:** What would happen if all threads used the tag 0?

In [ ]:
%%mpi -r -m -n 2 -e OMP_NUM_THREADS=4 -o code/hybrid-multiple.c
#include <stdio.h>
#include <mpi.h>
#include <omp.h>

int main(int argc, char **argv) {
    int provided, rank, size;

    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (provided < MPI_THREAD_MULTIPLE || size != 2) {
        if (rank == 0)
            fprintf(stderr, "requires MPI_THREAD_MULTIPLE and exactly 2 processes\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    int other = 1 - rank;
    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        int sendval = 100 * rank + tid, recvval;

        // the thread number as tag pairs thread t of one process with thread t of the other
        MPI_Sendrecv(&sendval, 1, MPI_INT, other, tid,
                     &recvval, 1, MPI_INT, other, tid, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

        printf("rank %d, thread %d: received %3d\n", rank, tid, recvval);
    }

    MPI_Finalize();
    return 0;
}

Further rules for `MPI_THREAD_MULTIPLE`:

* **Ordering:** concurrent MPI calls of different threads behave as if they were executed in some sequential order — any order.
  Within each thread, the order is maintained.
* **Blocking:** a blocking call blocks only the calling thread; the other threads continue and may call MPI.
* Preventing conflicting calls — e.g., two threads waiting on the same request — is the user's responsibility.

### Partitioned Communication

Often, the threads produce the parts of **one** message — e.g. a halo buffer filled in a worksharing loop.
Both patterns so far have a drawback then:

* **funneled:** the main thread can send only after *all* threads have finished — a barrier, and no overlap,
* **multiple:** every thread sends its part as a separate message — many small messages, matched one by one.

This is the main use case of **partitioned communication** (MPI 4.0), introduced in the point-to-point notebook: one persistent message whose partitions are marked ready one by one.

```c
MPI_Psend_init(buf, partitions, count, datatype, dest,   tag, comm, info, &request);  // set up once
MPI_Precv_init(buf, partitions, count, datatype, source, tag, comm, info, &request);
MPI_Start(&request);                         // in every iteration ...
MPI_Pready(partition, request);              // sender: partition is complete
MPI_Parrived(request, partition, &flag);     // receiver: has it arrived?
MPI_Wait(&request, MPI_STATUS_IGNORE);       // ... complete the whole message
```

What makes it fit threads: `MPI_Pready()` may be called **by any thread, in any order** — every thread releases its partition as soon as it has filled it, without a barrier.
Likewise, every receiving thread can wait with `MPI_Parrived()` for just the partition it works on.

In the following cell, process 0 fills a buffer of 8 partitions in an OpenMP loop; every thread marks a partition ready as soon as it has filled it.
On process 1, every thread waits only for the partition it works on.
Since several threads call MPI concurrently, the program requests `MPI_THREAD_MULTIPLE`.

> 🤔 **Predict:** Process 1 starts summing before the whole message has arrived. Why is that safe?

In [ ]:
%%mpi -r -m -n 2 -e OMP_NUM_THREADS=4 -o code/hybrid-partitioned.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>
#include <omp.h>

#define NPART 8                                 // number of partitions
#define COUNT 100000                            // elements per partition

int main(int argc, char **argv) {
    int provided, rank, size;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    if (provided < MPI_THREAD_MULTIPLE || size != 2) {
        if (rank == 0)
            fprintf(stderr, "requires MPI_THREAD_MULTIPLE and exactly 2 processes\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    double *buf = malloc(NPART * COUNT * sizeof(double));
    MPI_Request req;
    if (rank == 0)                              // one message of NPART partitions, set up once
        MPI_Psend_init(buf, NPART, COUNT, MPI_DOUBLE, 1, 0, MPI_COMM_WORLD, MPI_INFO_NULL, &req);
    else
        MPI_Precv_init(buf, NPART, COUNT, MPI_DOUBLE, 0, 0, MPI_COMM_WORLD, MPI_INFO_NULL, &req);

    for (int iter = 0; iter < 3; ++iter) {
        MPI_Start(&req);                        // start the operation: no data is sent yet

        if (rank == 0) {
            // producer: fill the partitions, mark each one ready as soon as it is complete
            #pragma omp parallel for schedule(dynamic)
            for (int p = 0; p < NPART; ++p) {
                for (int i = 0; i < COUNT; ++i)
                    buf[p * COUNT + i] = iter + p;
                MPI_Pready(p, req);             // partition p may be sent now
            }
        } else {
            // consumer: every thread waits only for the partition it works on
            double sum = 0.0;
            #pragma omp parallel for schedule(dynamic) reduction(+:sum)
            for (int p = 0; p < NPART; ++p) {
                int arrived = 0;
                while (!arrived)
                    MPI_Parrived(req, p, &arrived);
                for (int i = 0; i < COUNT; ++i)
                    sum += buf[p * COUNT + i];
            }
            double expected = (double)COUNT * (NPART * iter + NPART * (NPART - 1) / 2);
            printf("iteration %d: sum = %.0f, expected %.0f\n", iter, sum, expected);
        }

        MPI_Wait(&req, MPI_STATUS_IGNORE);      // complete the whole message
    }

    MPI_Request_free(&req);
    free(buf);
    MPI_Finalize();
    return 0;
}

`MPI_Parrived()` returns `true` only when the data of the partition is in the receive buffer — reading it is safe, even if other partitions are still in transit.
Message matching happens once for the whole operation, not per partition.

Compared with the funneled version, no thread waits for the others before its data can leave; compared with one message per thread, there is a single request and a single match.
As noted in the point-to-point notebook, whether partitions are actually sent early depends on the MPI implementation.
A strong motivation is GPUs: GPU kernels can mark partitions ready while they run.

## Pitfall: Collectives from Several Threads

Collectives have no tags: on one communicator, they are matched **in the order in which they are called** (see [Collective Communication](collectives.ipynb)).
If two threads of a process call collectives on the same communicator, the order depends on the thread scheduling — and may differ between processes:

![Process 0 calls MPI_Bcast on thread 0 before MPI_Barrier on thread 1; process 1 calls them in the opposite order. MPI matches the broadcast of process 0 with the barrier of process 1, which is erroneous](img/collective-order.svg)

The following cell provokes this with two broadcasts from rank 0, so that the effect becomes visible: thread 0 broadcasts `a`, thread 1 broadcasts `b`.
The threads are delayed so that process 0 broadcasts `a` first, and process 1 `b` first.

> 🤔 **Predict:** Which values do the processes print?

In [ ]:
%%mpi -r -m -n 2 -T 5
#include <stdio.h>
#include <mpi.h>
#include <omp.h>

static void busy_wait(double seconds) {
    double t0 = omp_get_wtime();
    while (omp_get_wtime() - t0 < seconds)
        ;
}

int main(int argc, char **argv) {
    int provided, rank;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    if (provided < MPI_THREAD_MULTIPLE) {
        if (rank == 0)
            fprintf(stderr, "requires MPI_THREAD_MULTIPLE\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    int a = rank == 0 ? 1 : 0, b = rank == 0 ? 2 : 0;
    #pragma omp parallel num_threads(2)
    {
        int tid = omp_get_thread_num();
        // process 0: a first; process 1: b first
        if ((rank == 0 && tid == 1) || (rank == 1 && tid == 0))
            busy_wait(0.2);

        if (tid == 0)
            MPI_Bcast(&a, 1, MPI_INT, 0, MPI_COMM_WORLD);
        else
            MPI_Bcast(&b, 1, MPI_INT, 0, MPI_COMM_WORLD);
    }
    printf("rank %d: a = %d, b = %d\n", rank, a, b);

    MPI_Finalize();
    return 0;
}

The first broadcast of process 0 is matched with the first broadcast of process 1 — `a` and `b` are swapped on process 1, and MPI cannot detect it:
the standard leaves it to the user to order the collective calls of several threads on the same communicator.
If the two collectives differ — like the broadcast and the barrier in the figure — the program is erroneous and may hang, crash or silently produce garbage.

Synchronizing the threads so that the order is the same on all processes would fix it, but defeats the purpose of running them concurrently.
The clean solution: **one communicator per thread**.
Collectives on different communicators are independent, so the order across them doesn't matter.
`MPI_Comm_dup()` creates a copy of a communicator with the same processes — collectively, so it must be called outside the parallel region (or in the same order by all threads):

In [ ]:
%%mpi -r -m -n 2 -T 5 -o code/hybrid-comm-per-thread.c
#include <stdio.h>
#include <mpi.h>
#include <omp.h>

static void busy_wait(double seconds) {
    double t0 = omp_get_wtime();
    while (omp_get_wtime() - t0 < seconds)
        ;
}

int main(int argc, char **argv) {
    int provided, rank;
    MPI_Init_thread(&argc, &argv, MPI_THREAD_MULTIPLE, &provided);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    if (provided < MPI_THREAD_MULTIPLE) {
        if (rank == 0)
            fprintf(stderr, "requires MPI_THREAD_MULTIPLE\n");
        MPI_Abort(MPI_COMM_WORLD, 1);
    }

    // one communicator per thread, created before the parallel region
    MPI_Comm comms[2];
    for (int t = 0; t < 2; ++t)
        MPI_Comm_dup(MPI_COMM_WORLD, &comms[t]);

    int a = rank == 0 ? 1 : 0, b = rank == 0 ? 2 : 0;
    #pragma omp parallel num_threads(2)
    {
        int tid = omp_get_thread_num();
        if ((rank == 0 && tid == 1) || (rank == 1 && tid == 0))
            busy_wait(0.2);

        if (tid == 0)
            MPI_Bcast(&a, 1, MPI_INT, 0, comms[0]);   // each thread on its own communicator
        else
            MPI_Bcast(&b, 1, MPI_INT, 0, comms[1]);
    }
    printf("rank %d: a = %d, b = %d\n", rank, a, b);

    for (int t = 0; t < 2; ++t)
        MPI_Comm_free(&comms[t]);
    MPI_Finalize();
    return 0;
}

One communicator per thread is also the key to **performance** with `MPI_THREAD_MULTIPLE` — for point-to-point communication, too:

* MPICH (since 4.0) can map communicators to separate *virtual communication interfaces* (VCIs), each with its own lock and network resources: set `MPIR_CVAR_CH4_NUM_VCIS` to the number of threads.
  Intel MPI offers the same with its thread-split model (`I_MPI_THREAD_SPLIT=1`).
  The threads then no longer serialize on one lock in the library — but measure: very many VCIs are not always faster.
* MPI 4.0 **info assertions** tell the library which features a communicator will not use, so that it can simplify message matching:

```c
MPI_Info info;
MPI_Info_create(&info);
MPI_Info_set(info, "mpi_assert_no_any_source", "true");   // no MPI_ANY_SOURCE on this communicator
MPI_Info_set(info, "mpi_assert_no_any_tag", "true");      // no MPI_ANY_TAG
for (int t = 0; t < nthreads; ++t)
    MPI_Comm_dup_with_info(MPI_COMM_WORLD, info, &comms[t]);
MPI_Info_free(&info);
```

Further assertions are `mpi_assert_exact_length` and `mpi_assert_allow_overtaking`.
A library may ignore them — but a program that violates an assertion it made is erroneous.

## Compiling, Starting and Pinning

### Compiling

* Use the OpenMP flag of the compiler — `-fopenmp` (GCC, clang, Intel oneAPI `icx`), `-qopenmp` (classic Intel), `-mp` (NVIDIA HPC SDK) — together with the MPI compiler wrapper, e.g. `mpicc -fopenmp` or `mpiicx -qopenmp`.
* The wrapper links the MPI library; current MPI libraries are thread-safe by default.

### Nomenclature

![A dual-socket node with 4 cores per socket in three configurations: pure MPI with 8 processes; fully hybrid with one process of 8 threads; mixed mode with one process of 4 threads per socket](img/hybrid-modes.svg)

* **Pure MPI:** one process per core, no threads.
* **Fully hybrid:** one process per node, OpenMP threads on all cores.
* **Mixed mode:** several processes per node, several threads per process — typically one process per ccNUMA domain (in the figure: per socket), which avoids most ccNUMA problems of threading (see below).

Current CPUs often have several ccNUMA domains per socket (Intel sub-NUMA clustering, AMD NPS modes), and AMD CPUs have one L3 cache per group of up to 8 cores.
One process per ccNUMA domain — or even per L3 cache — is then a good starting point for mixed mode.
A program can find out which of its processes share a ccNUMA domain with `MPI_Comm_split_type(…, MPI_COMM_TYPE_HW_GUIDED, …)` and the info key `mpi_hw_resource_type` (MPI 4.0; the values, such as `"NUMANode"`, are implementation-defined), see Communicators, Groups and Shared-Memory Windows.

### Starting and Pinning

Starting hybrid programs is **highly non-portable** — consult the documentation of your system.
The points to get right:

* start fewer processes than cores per node, and reserve the cores for the threads of each process,
* make sure `OMP_NUM_THREADS` (and other variables) reach all processes,
* **pin** processes and threads to cores, so that they don't migrate and all cores are used — use the standard OpenMP variables `OMP_PLACES=cores` and `OMP_PROC_BIND=close` (or `spread`) for the threads, as introduced in the OpenMP primer.

On a node with 2 sockets × 32 cores and 4 ccNUMA domains of 16 cores, for example, on 2 nodes:

| | Fully hybrid: 1 process × 64 threads per node | Mixed mode: 4 processes × 16 threads per node |
|---|---|---|
| **Slurm** | `srun --nodes=2 --ntasks-per-node=1 --cpus-per-task=64 --cpu-bind=cores ./a.out` | `srun --nodes=2 --ntasks-per-node=4 --cpus-per-task=16 --cpu-bind=cores ./a.out` |
| **Open MPI** | `mpirun -n 2 --map-by ppr:1:node:PE=64 --bind-to core ./a.out` | `mpirun -n 8 --map-by ppr:1:numa:PE=16 --bind-to core ./a.out` |
| **Intel MPI** | `mpirun -n 2 -ppn 1 -genv I_MPI_PIN_DOMAIN omp ./a.out` | `mpirun -n 8 -ppn 4 -genv I_MPI_PIN_DOMAIN omp ./a.out` |
| **LIKWID** | `likwid-mpirun -np 2 -pin N:0-63 ./a.out` | `likwid-mpirun -np 8 -pin M0:0-15_M1:0-15_M2:0-15_M3:0-15 ./a.out` |

with `OMP_NUM_THREADS=64` or `16` and `OMP_PLACES=cores OMP_PROC_BIND=close` exported.

* **Slurm:** in versions 22.05 to 23.02, `srun` did not inherit `--cpus-per-task` from the `#SBATCH` options; since 23.11, it does again.
  Passing `--cpus-per-task` to `srun` explicitly works with all versions.
  Set `OMP_NUM_THREADS=$SLURM_CPUS_PER_TASK` in the job script.
* **Open MPI 5** calls sockets `package`; `numa` and `l3cache` map one process per ccNUMA domain or L3 cache.
  `--report-bindings` shows where the processes are bound.
* **Intel MPI:** `I_MPI_PIN_DOMAIN=omp` gives every process a domain of `OMP_NUM_THREADS` cores; `numa` one ccNUMA domain per process.
  `I_MPI_DEBUG=4` prints the pinning.
* Intel-specific variables such as `KMP_AFFINITY` work only with the Intel runtime; prefer the standard `OMP_PLACES`/`OMP_PROC_BIND`.
* `likwid-mpirun` is the simplest interface and reports where everything runs — recommended where available.
* On macOS, processes and threads cannot be pinned at all; pinning experiments need Linux.
* **Pin the threads only if the processes are pinned, too.** With `--bind-to none` — as with `-m` in this notebook — every process may run on all cores.
  Every process then computes the placement of its threads independently from the same list of places, and with `OMP_PROC_BIND` set, the threads of different processes may all end up on the same few cores.
  For time measurements on Linux, replace the launcher flags of the cell, e.g. `-L "--map-by slot:PE=2 --bind-to core" -e OMP_NUM_THREADS=2 OMP_PLACES=cores OMP_PROC_BIND=close`.

### Where Am I Running?

The following cell reports, for every process and thread, the host, the CPU it is currently running on and its **affinity mask** — the CPUs it is allowed to run on.
Both are only available on Linux (`sched_getcpu()`, `sched_getaffinity()`); elsewhere they show `n/a`.
The data is collected on rank 0 and printed in order.
Run it on a cluster with different pinning options and compare.

In [ ]:
%%mpi -m -n 2 -e OMP_NUM_THREADS=3 -f D_GNU_SOURCE -i sched.h string.h -o code/hybrid-where.c
#define MASK_LEN 128

char host[MPI_MAX_PROCESSOR_NAME] = { 0 };
int len;
MPI_Get_processor_name(host, &len);

// every thread records where it runs and where it may run
const int nthreads = omp_get_max_threads();
int  *cpus  = malloc(nthreads * sizeof(int));
char *masks = calloc(nthreads, MASK_LEN);

#pragma omp parallel
{
    int tid = omp_get_thread_num();
    char *mask = masks + tid * MASK_LEN;
#ifdef __linux__
    cpus[tid] = sched_getcpu();
    cpu_set_t set;
    sched_getaffinity(0, sizeof(set), &set);       // 0: the calling thread
    int pos = 0;
    for (int c = 0; c < CPU_SETSIZE && pos < MASK_LEN - 8; ++c)
        if (CPU_ISSET(c, &set))
            pos += snprintf(mask + pos, MASK_LEN - pos, "%s%d", pos ? "," : "", c);
#else
    cpus[tid] = -1;
    strcpy(mask, "n/a");
#endif
}

// collect everything on rank 0 (assumes the same number of threads on all processes)
char *allhosts = NULL, *allmasks = NULL;
int  *allcpus  = NULL;
if (rank == 0) {
    allhosts = malloc(size * MPI_MAX_PROCESSOR_NAME);
    allcpus  = malloc(size * nthreads * sizeof(int));
    allmasks = malloc(size * nthreads * MASK_LEN);
}
MPI_Gather(host, MPI_MAX_PROCESSOR_NAME, MPI_CHAR, allhosts, MPI_MAX_PROCESSOR_NAME, MPI_CHAR, 0, MPI_COMM_WORLD);
MPI_Gather(cpus, nthreads, MPI_INT, allcpus, nthreads, MPI_INT, 0, MPI_COMM_WORLD);
MPI_Gather(masks, nthreads * MASK_LEN, MPI_CHAR, allmasks, nthreads * MASK_LEN, MPI_CHAR, 0, MPI_COMM_WORLD);

if (rank == 0) {
    const char *places = getenv("OMP_PLACES"), *bind = getenv("OMP_PROC_BIND");
    printf("OMP_PLACES=%s, OMP_PROC_BIND=%s\n", places ? places : "(unset)", bind ? bind : "(unset)");
    for (int r = 0; r < size; ++r)
        for (int t = 0; t < nthreads; ++t) {
            int cpu = allcpus[r * nthreads + t];
            char cpustr[16];
            snprintf(cpustr, sizeof(cpustr), "%d", cpu);
            printf("rank %d on %s, thread %d: on CPU %s, may run on CPUs %s\n", r, allhosts + r * MPI_MAX_PROCESSOR_NAME,
                   t, cpu < 0 ? "n/a" : cpustr, allmasks + (r * nthreads + t) * MASK_LEN);
        }
}

free(cpus);
free(masks);
free(allhosts);
free(allcpus);
free(allmasks);

### ccNUMA and First Touch

On a ccNUMA node, every memory page belongs to one ccNUMA domain; accessing a page in another domain is slower and competes for that domain's memory bandwidth.
Linux places a page in the domain of the thread that **first writes** it — not of the thread that called `malloc()`.
For threaded code, this means:

* Initialize the data **in parallel**, with the **same** distribution of the iterations as in the computation, so that every thread touches the pages it will work on.
  Serial initialization — including the zeroing that `calloc()` may do — puts all pages into the domain of the main thread, and all threads compete for its memory interface.
* This works only if the threads are pinned; otherwise they may migrate away from their pages.
* Pure MPI and mixed mode with one process per ccNUMA domain avoid the problem: every process touches only its own data.

```c
double *u = malloc(n * sizeof(double));        // reserves address space only, no pages yet
#pragma omp parallel for schedule(static)
for (int i = 0; i < n; ++i)
    u[i] = 0.0;                                // first touch: the page goes to this thread's domain

#pragma omp parallel for schedule(static)      // same schedule: every thread works on "its" pages
for (int i = 0; i < n; ++i)
    …
```

Exercise 2 puts this into practice.

## Load Balancing with OpenMP

A benefit of threads within a process: work can be redistributed **dynamically** among them, at almost no cost.
Consider the multiplication of an upper triangular matrix with a vector: row `i` has `N - i` non-zero elements, so the work per row decreases linearly.

```c
for (int i = 0; i < N; ++i) {
    double sum = 0.0;
    for (int j = i; j < N; ++j)
        sum += mat[i * N + j] * b[j];
    c[i] = sum;
}
```

The default `schedule(static)` gives every thread one contiguous block of rows — thread 0 gets the longest rows and has the most work.
The lecture measured this with hardware performance counters (LIKWID); here, every thread simply measures the time it spends working, in the same loop, for four schedules:

| Schedule | Distribution of the iterations |
|---|---|
| `static` | one contiguous block per thread |
| `static,16` | blocks of 16 iterations, assigned round-robin |
| `dynamic,16` | blocks of 16, assigned to whichever thread is free |
| `guided` | dynamic, with block sizes decreasing from large to small |

The loop uses `schedule(runtime)` and sets the schedule with `omp_set_schedule()`; `nowait` with an explicit barrier after the timing makes sure that waiting time is not counted as work.

> 🤔 **Predict:** How does the work of the 4 threads compare with `schedule(static)`?

In [ ]:
%%c -m -e OMP_NUM_THREADS=4 -O imbalance -o code/hybrid-triangular-mvm.c
const int N = 3000, ROUNDS = 20;

double *mat = malloc((size_t)N * N * sizeof(double));
double *b = malloc(N * sizeof(double)), *c = malloc(N * sizeof(double));
for (size_t k = 0; k < (size_t)N * N; ++k)
    mat[k] = 1.0 / (1 + k % 7);
for (int i = 0; i < N; ++i)
    b[i] = 1.0;

const char *names[]  = { "static", "static,16", "dynamic,16", "guided" };
omp_sched_t kinds[]  = { omp_sched_static, omp_sched_static, omp_sched_dynamic, omp_sched_guided };
int chunks[]         = { 0, 16, 16, 0 };    // 0: the default chunk size
const int nthreads   = omp_get_max_threads();
double *busy = malloc(nthreads * sizeof(double));

printf("%-12s %10s", "schedule", "total[ms]");
for (int t = 0; t < nthreads; ++t)
    printf("  thread%d[ms]", t);
printf("\n");

for (int s = 0; s < 4; ++s) {
    omp_set_schedule(kinds[s], chunks[s]);  // used by schedule(runtime)
    double t0 = omp_get_wtime();

    #pragma omp parallel
    {
        double mine = 0.0;
        for (int k = 0; k < ROUNDS; ++k) {
            double start = omp_get_wtime();
            #pragma omp for schedule(runtime) nowait
            for (int i = 0; i < N; ++i) {
                double sum = 0.0;
                for (int j = i; j < N; ++j)
                    sum += mat[(size_t)i * N + j] * b[j];
                c[i] = sum;
            }
            mine += omp_get_wtime() - start;    // the time this thread spent working
            #pragma omp barrier
        }
        busy[omp_get_thread_num()] = mine;
    }

    double total = omp_get_wtime() - t0, checksum = 0.0;
    for (int i = 0; i < N; ++i)
        checksum += c[i];                       // use the result, so the compiler can't drop the loop
    printf("%-12s %10.1f", names[s], 1e3 * total);
    for (int t = 0; t < nthreads; ++t)
        printf(" %12.1f", 1e3 * busy[t]);
    printf("   (checksum %.6e)\n", checksum);
}

free(mat);
free(b);
free(c);
free(busy);

In [ ]:
# plot the busy time of every thread per schedule (requires matplotlib)
import matplotlib.pyplot as plt

surface, ink, muted, grid, axis = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#a3a29c'
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']

rows = [line.split() for line in imbalance.splitlines() if line and not line.startswith('schedule')]
rows = [r for r in rows if len(r) > 2 and r[1].replace('.', '').isdigit()]
schedules = [r[0] for r in rows]
nthreads = len(rows[0]) - 4                 # name, total, threads..., (checksum ...)
busy = [[float(v) for v in r[2:2 + nthreads]] for r in rows]

fig, ax = plt.subplots(figsize=(9, 4.4), facecolor=surface)
ax.set_facecolor(surface)
width = 0.8 / nthreads
for t in range(nthreads):
    xs = [s + (t - (nthreads - 1) / 2) * width for s in range(len(schedules))]
    ax.bar(xs, [b[t] for b in busy], width * 0.92, color=colors[t % len(colors)], label=f'thread {t}')
for s, r in enumerate(rows):
    ax.annotate(f'total {float(r[1]):.0f} ms', (s, max(busy[s])), xytext=(0, 6), textcoords='offset points',
                ha='center', fontsize=10, color=ink)
ax.set_xticks(range(len(schedules)), [f'schedule({s})' for s in schedules], color=muted)
ax.set_title('Busy time per thread [ms] — triangular matrix-vector multiplication', loc='left', fontsize=12, color=ink)
ax.grid(True, axis='y', color=grid, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=muted)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
for side in ('left', 'bottom'):
    ax.spines[side].set_color(axis)
ax.legend(frameon=False, labelcolor=ink, ncols=nthreads, loc='upper right')
ax.set_ylim(0, max(max(b) for b in busy) * 1.25)
fig.tight_layout()
plt.show()

With `schedule(static)`, the threads that finish early wait at the barrier — the total time is that of the slowest thread.
Smaller blocks assigned round-robin already balance the work almost perfectly; dynamic schedules adapt even to imbalance that is not known in advance — at the cost of some scheduling overhead.
With pure MPI, such a rebalancing would require moving data between processes.

## Pros and Cons of Hybrid Programming

Many arguments for hybrid programming are lore rather than fact — and several have weakened, since MPI libraries have learned to exploit the node themselves:

| Claim | Reality |
|---|---|
| It "fits" the hierarchical hardware: threads within a node, MPI between nodes. | Not always. OpenMP opens its own can of worms: ccNUMA page placement, overheads, affinity. And MPI libraries exploit the hierarchy, too: messages within a node go through shared memory, and collectives are node-aware (e.g. Open MPI's `han`). |
| It reduces the communication volume and the number of messages. | Not always. MPI-only codes can be optimized too, and the inter-node volume may be the same. |
| OpenMP is more lightweight and thus more efficient than MPI within a node. | Not generally. It depends entirely on the code — compare the cost of a full-node OpenMP barrier with an MPI latency. Single-copy transfers within a node (XPMEM, CMA, KNEM) have made large intra-node messages cheap. |
| Many processes slow down start-up and collectives. | Much less than they used to: start-up via PMIx scales well, and node-aware collectives combine the data within a node first. |
| Every process wastes memory on MPI buffers and connections. | Much less than it used to: connections are set up on demand, and connectionless transports (e.g. UCX DC) need no state per peer. Replicated *application* data remains an issue. |

The **real advantages**:

* **Load balancing** becomes easier with dynamic loop schedules or tasks (see above).
* **Less memory** for replicated application data — tables, ghost layers: it is stored once per process instead of once per core.
  For the multi-zone NAS benchmarks BT-MZ and SP-MZ, 8 threads per process reduced the memory footprint more than fivefold (Shan et al., *CUG 2010*).
  Pure MPI can achieve the same without threads: shared-memory windows (`MPI_Win_allocate_shared()`) store such data once per node or ccNUMA domain.
* **GPUs:** with one MPI process per GPU, a node with 4 GPUs and 64 cores runs only 4 processes — the other cores can only be used by threads.
* **Communication overlap:** a dedicated thread can communicate while the others compute (Exercise 3).
  It is the most portable way to asynchronous progress, but costs a core.
  Alternatives are progress threads of the MPI library (`MPIR_CVAR_ASYNC_PROGRESS=1` in MPICH, `I_MPI_ASYNC_PROGRESS=1` in Intel MPI) and networks that offload message matching and transfers.
* **More parallelism:** OpenMP can exploit levels of parallelism where MPI cannot go, e.g. nested loops or tasks.

The **problems**:

* **Affinity** control becomes more complex: OpenMP standardizes the pinning of threads, but the pinning of processes at start-up is not standardized — and both have to fit together.
* **Two levels of parallelism** mean two levels of overhead — and two levels of Amdahl's law: if a fraction $s$ of the code is serial in every process and a fraction $s_t$ of the rest is not threaded, the speedup with $p$ processes of $t$ threads is at most $1 / \left(s + (1-s)\left(s_t + (1-s_t)/t\right)/p\right)$.
* **Message rate:** with the funneled pattern, one thread per process does all the communication.
  With small messages, one core may not be able to saturate a modern network link (200–400 Gbit/s).
  Remedies: several processes per node, `MPI_THREAD_MULTIPLE` with one communicator per thread, or partitioned communication.
* **Interoperability:** OpenMP and MPI know little about each other; splitting off a thread for communication entails a lot of complexity, and the thread support level must be right.
  Newer features help: OpenMP tasks with a `detach` clause (OpenMP 5.0) can complete when an MPI request completes (via `omp_fulfill_event()`, used e.g. by the TAMPI library), and partitioned communication lets threads contribute to one message.
* Threading may expose new bottlenecks: in the MiniMD proxy app, the variant with random writes in the innermost loop scaled as well as MPI with processes, but much worse with OpenMP threads.

Pure MPI, in contrast, is simpler to program, has no thread-safety concerns and no ccNUMA placement problems, and only one level of overhead — but replicated data can get out of hand (unless shared-memory windows are used), load balancing is hard, and overlap is not guaranteed.

### Which Problem, Which Remedy?

| Problem | Options |
|---|---|
| Replicated data exhausts the memory | threads; or pure MPI with shared-memory windows |
| Load imbalance within a node | OpenMP dynamic schedules or tasks |
| No overlap of communication and computation | progress threads of the MPI library, network offload; a dedicated communication thread |
| Message rate too low | several processes per node; `MPI_THREAD_MULTIPLE` with one communicator per thread; partitioned communication |
| One process per GPU leaves CPU cores idle | threads for the host part |
| Scalability limits of pure MPI at very large process counts | fewer processes with threads — but measure first: MPI libraries have improved a lot |

> **Conclusion:** don't be fooled by lore and anecdotes.
> The benefit of going hybrid — starting from an MPI code — depends heavily on the code.
> Consider it only if you have a concrete problem it solves — and check whether pure MPI with current library features solves it, too.

## Exercises

### Exercise 1: Hybrid π

Parallelize the computation of $\pi$ on two levels: distribute the intervals among the MPI processes as in [MPI in a Nutshell](mpi-in-a-nutshell.ipynb), parallelize the loop of every process with OpenMP, and combine the results with `MPI_Reduce()`.
Then compare the runtime for the same total number of cores, e.g. `-n 4` with 1 thread, `-n 2` with 2 threads, and `-n 1` with 4 threads (on Linux, pin processes and threads for this, see Starting and Pinning).

<details>
<summary>Solution</summary>

```c
double sum = 0.0;
#pragma omp parallel for reduction(+:sum)
for (int64_t i = start; i < end; ++i) {
    double x = (i + 0.5) * h;
    sum += 4.0 / (1.0 + x * x);
}
sum *= h;

double pi;
MPI_Reduce(&sum, &pi, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
if (rank == 0)
    printf("%d processes x %d threads: pi = %.15f, error %.2e\n", size, omp_get_max_threads(), pi, pi - 3.14159265358979323846);
```

The MPI call is outside the parallel region, so `MPI_THREAD_FUNNELED` is sufficient.
For such an embarrassingly parallel loop, all combinations perform about the same.
</details>

In [ ]:
%%mpi -m -n 2 -e OMP_NUM_THREADS=2 -t
const int64_t N = 1000000000;
const double h = 1.0 / N;

int64_t start =  rank      * N / size;
int64_t end   = (rank + 1) * N / size;

// TODO: compute the partial sum of this process with OpenMP threads,
//       combine the partial sums with MPI_Reduce() and print pi on rank 0

### Exercise 2: Hybrid Laplace Solver

The cell contains the Laplace solver of Derived Data Types and Virtual Topologies (Exercise 4) on a 256 × 256 grid, with the ghost cell exchange in place; it runs a fixed number of 2000 iterations.
Parallelize the Jacobi update with OpenMP, and initialize the arrays in parallel with the same distribution of the rows (first touch).
Make sure that the last change and the error are the same as without threads.

Then compare the runtime for 4 cores in total: `-n 4` × 1 thread, `-n 2` × 2 threads, `-n 1` × 4 threads (on Linux, pin processes and threads for this, see Starting and Pinning).

<details>
<summary>Solution</summary>

Add a directive in front of the initialization loop, with the same (default `static`) schedule over `j` as the update loop:

```c
#pragma omp parallel for
for (int j = 0; j <= jmaxLocal + 1; ++j)
    …
```

and one in front of the update loop:

```c
#pragma omp parallel for reduction(max:local_change)
for (int j = 1; j <= jmaxLocal; ++j)
    …
```

The halo exchange stays outside the parallel region and is done by the main thread only — `MPI_THREAD_FUNNELED`.
A `max` reduction is exact, so the result does not depend on the number of threads.

Don't be surprised if pure MPI is fastest on your machine: the threads are forked and joined in every iteration, and on a laptop all threads share one memory interface anyway — a perfect example for "it depends on the code".
On a node with several ccNUMA domains, the parallel first touch decides whether the fully hybrid run can use the memory bandwidth of all domains.
</details>

In [ ]:
%%mpi -m -n 2 -e OMP_NUM_THREADS=2 -t
const int imax = 256, jmax = 256;               // global number of inner grid points
const double tol = 1e-12;                       // not reached: a fixed number of iterations,
const int maxiter = 2000;                       // so that runtimes are comparable

int dims[2] = { 0, 0 }, periods[2] = { 0, 0 }, coords[2];
MPI_Dims_create(size, 2, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, 2, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, 2, coords);
int left, right, bottom, top;
MPI_Cart_shift(comm, 0, 1, &left, &right);
MPI_Cart_shift(comm, 1, 1, &bottom, &top);

#define SIZE_OF(c, n, p)  ((n) / (p) + ((c) < (n) % (p) ? 1 : 0))
#define START_OF(c, n, p) ((c) * ((n) / (p)) + ((c) < (n) % (p) ? (c) : (n) % (p)))
const int imaxLocal = SIZE_OF(coords[0], imax, dims[0]), ioff = START_OF(coords[0], imax, dims[0]);
const int jmaxLocal = SIZE_OF(coords[1], jmax, dims[1]), joff = START_OF(coords[1], jmax, dims[1]);
const int ld = imaxLocal + 2;
const double h = 1.0 / (imax + 1);

double *u    = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
double *unew = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
#define U(i, j)    u[(j) * ld + (i)]
#define UNEW(i, j) unew[(j) * ld + (i)]
#define X(i) ((ioff + (i)) * h)
#define Y(j) ((joff + (j)) * h)

// TODO: initialize in parallel, with the same distribution as the update (first touch)
for (int j = 0; j <= jmaxLocal + 1; ++j)
    for (int i = 0; i <= imaxLocal + 1; ++i)
        U(i, j) = UNEW(i, j) = 0.0;

for (int j = 0; j <= jmaxLocal + 1; ++j) {
    if (left  == MPI_PROC_NULL) U(0, j)             = UNEW(0, j)             = X(0) + Y(j);
    if (right == MPI_PROC_NULL) U(imaxLocal + 1, j) = UNEW(imaxLocal + 1, j) = X(imaxLocal + 1) + Y(j);
}
for (int i = 0; i <= imaxLocal + 1; ++i) {
    if (bottom == MPI_PROC_NULL) U(i, 0)             = UNEW(i, 0)             = X(i) + Y(0);
    if (top    == MPI_PROC_NULL) U(i, jmaxLocal + 1) = UNEW(i, jmaxLocal + 1) = X(i) + Y(jmaxLocal + 1);
}

MPI_Datatype column, row;
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);
MPI_Type_commit(&column);
MPI_Type_commit(&row);

int iter;
double change = 1.0;
for (iter = 0; iter < maxiter && change > tol; ++iter) {
    MPI_Request requests[8];
    MPI_Irecv(&U(1, 0),             1, row,    bottom, 0, comm, &requests[0]);
    MPI_Irecv(&U(1, jmaxLocal + 1), 1, row,    top,    1, comm, &requests[1]);
    MPI_Irecv(&U(0, 1),             1, column, left,   2, comm, &requests[2]);
    MPI_Irecv(&U(imaxLocal + 1, 1), 1, column, right,  3, comm, &requests[3]);
    MPI_Isend(&U(1, jmaxLocal),     1, row,    top,    0, comm, &requests[4]);
    MPI_Isend(&U(1, 1),             1, row,    bottom, 1, comm, &requests[5]);
    MPI_Isend(&U(imaxLocal, 1),     1, column, right,  2, comm, &requests[6]);
    MPI_Isend(&U(1, 1),             1, column, left,   3, comm, &requests[7]);
    MPI_Waitall(8, requests, MPI_STATUSES_IGNORE);

    double local_change = 0.0;
    // TODO: parallelize the update with OpenMP
    for (int j = 1; j <= jmaxLocal; ++j)
        for (int i = 1; i <= imaxLocal; ++i) {
            UNEW(i, j) = 0.25 * (U(i - 1, j) + U(i + 1, j) + U(i, j - 1) + U(i, j + 1));
            double diff = UNEW(i, j) - U(i, j);
            if (diff < 0.0) diff = -diff;
            if (diff > local_change) local_change = diff;
        }
    double *tmp = u;
    u = unew;
    unew = tmp;
    MPI_Allreduce(&local_change, &change, 1, MPI_DOUBLE, MPI_MAX, comm);
}

double local_err = 0.0, err;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i) {
        double diff = U(i, j) - (X(i) + Y(j));
        if (diff < 0.0) diff = -diff;
        if (diff > local_err) local_err = diff;
    }
MPI_Reduce(&local_err, &err, 1, MPI_DOUBLE, MPI_MAX, 0, comm);
if (rank == 0)
    printf("%d x %d processes x %d threads: %d iterations, last change %.6e, max. error %.6e\n",
           dims[0], dims[1], omp_get_max_threads(), iter, change, err);

MPI_Type_free(&column);
MPI_Type_free(&row);
MPI_Comm_free(&comm);
free(u);
free(unew);

### Exercise 3: Overlapping Communication with a Dedicated Thread *(advanced)*

In Exercise 2, all threads wait while the main thread exchanges the ghost cells.
Overlap the exchange with computation:

* Open the parallel region around exchange **and** update.
* Thread 0 does the halo exchange (`MPI_THREAD_FUNNELED` is still sufficient, since it is the main thread), while the **other** threads update the inner cells that do not need ghost cells (rows and columns 2 … `max - 1`); distribute these rows among the other threads by hand, e.g. round-robin.
* After a barrier, all threads update the cells next to the ghost cells (rows 1 and `jmaxLocal`, columns 1 and `imaxLocal`).

With a single thread, thread 0 must do both.
Check that the last change and the error don't change, and compare the runtime with Exercise 2.

<details>
<summary>Solution</summary>

```c
double local_change = 0.0;
#pragma omp parallel reduction(max:local_change)
{
    const int tid = omp_get_thread_num(), nt = omp_get_num_threads();

    if (tid == 0) {
        // halo exchange by the main thread, while the others work on the bulk
        MPI_Request requests[8];
        … the 8 calls and MPI_Waitall() of the exchange …
    }

    // the bulk: rows 2 ... jmaxLocal - 1, distributed round-robin among threads 1 ... nt - 1
    // (with a single thread, thread 0 does it after the exchange)
    const int workers = nt > 1 ? nt - 1 : 1, me = nt > 1 ? tid - 1 : 0;
    if (tid > 0 || nt == 1)
        for (int j = 2 + me; j <= jmaxLocal - 1; j += workers)
            for (int i = 2; i <= imaxLocal - 1; ++i)
                UPDATE(i, j);

    #pragma omp barrier                     // ghost cells and bulk are ready

    // the cells next to the ghost cells, shared among all threads:
    // the first and last row completely, the first and last column of the other rows
    #pragma omp for
    for (int j = 1; j <= jmaxLocal; ++j) {
        if (j == 1 || j == jmaxLocal) {
            for (int i = 1; i <= imaxLocal; ++i)
                UPDATE(i, j);
        } else {
            UPDATE(1, j);
            if (imaxLocal > 1)
                UPDATE(imaxLocal, j);
        }
    }
}
```

with a macro for the update of one cell:

```c
#define UPDATE(i, j) do {                                                               \
        UNEW(i, j) = 0.25 * (U((i) - 1, j) + U((i) + 1, j) + U(i, (j) - 1) + U(i, (j) + 1)); \
        double diff = UNEW(i, j) - U(i, j);                                             \
        if (diff < 0.0) diff = -diff;                                                   \
        if (diff > local_change) local_change = diff;                                   \
    } while (0)
```

Whether the overlap pays off depends on the ratio of communication to computation: on a single node with a small grid, the exchange is so fast that the extra synchronization — and the fact that only `nt - 1` threads work on the bulk — may cost more than it saves.
Try it with more processes or on a cluster.
</details>

In [ ]:
%%mpi -m -n 2 -e OMP_NUM_THREADS=3 -t
const int imax = 256, jmax = 256;               // global number of inner grid points
const double tol = 1e-12;                       // not reached: a fixed number of iterations,
const int maxiter = 2000;                       // so that runtimes are comparable

int dims[2] = { 0, 0 }, periods[2] = { 0, 0 }, coords[2];
MPI_Dims_create(size, 2, dims);
MPI_Comm comm;
MPI_Cart_create(MPI_COMM_WORLD, 2, dims, periods, 0, &comm);
MPI_Comm_rank(comm, &rank);
MPI_Cart_coords(comm, rank, 2, coords);
int left, right, bottom, top;
MPI_Cart_shift(comm, 0, 1, &left, &right);
MPI_Cart_shift(comm, 1, 1, &bottom, &top);

#define SIZE_OF(c, n, p)  ((n) / (p) + ((c) < (n) % (p) ? 1 : 0))
#define START_OF(c, n, p) ((c) * ((n) / (p)) + ((c) < (n) % (p) ? (c) : (n) % (p)))
const int imaxLocal = SIZE_OF(coords[0], imax, dims[0]), ioff = START_OF(coords[0], imax, dims[0]);
const int jmaxLocal = SIZE_OF(coords[1], jmax, dims[1]), joff = START_OF(coords[1], jmax, dims[1]);
const int ld = imaxLocal + 2;
const double h = 1.0 / (imax + 1);

double *u    = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
double *unew = malloc((size_t)ld * (jmaxLocal + 2) * sizeof(double));
#define U(i, j)    u[(j) * ld + (i)]
#define UNEW(i, j) unew[(j) * ld + (i)]
#define X(i) ((ioff + (i)) * h)
#define Y(j) ((joff + (j)) * h)

// first touch in parallel, as in Exercise 2
#pragma omp parallel for
for (int j = 0; j <= jmaxLocal + 1; ++j)
    for (int i = 0; i <= imaxLocal + 1; ++i)
        U(i, j) = UNEW(i, j) = 0.0;

for (int j = 0; j <= jmaxLocal + 1; ++j) {
    if (left  == MPI_PROC_NULL) U(0, j)             = UNEW(0, j)             = X(0) + Y(j);
    if (right == MPI_PROC_NULL) U(imaxLocal + 1, j) = UNEW(imaxLocal + 1, j) = X(imaxLocal + 1) + Y(j);
}
for (int i = 0; i <= imaxLocal + 1; ++i) {
    if (bottom == MPI_PROC_NULL) U(i, 0)             = UNEW(i, 0)             = X(i) + Y(0);
    if (top    == MPI_PROC_NULL) U(i, jmaxLocal + 1) = UNEW(i, jmaxLocal + 1) = X(i) + Y(jmaxLocal + 1);
}

MPI_Datatype column, row;
MPI_Type_vector(jmaxLocal, 1, ld, MPI_DOUBLE, &column);
MPI_Type_contiguous(imaxLocal, MPI_DOUBLE, &row);
MPI_Type_commit(&column);
MPI_Type_commit(&row);

int iter;
double change = 1.0;
for (iter = 0; iter < maxiter && change > tol; ++iter) {
    // TODO: overlap the exchange (thread 0) with the update of the bulk (the other threads)
    MPI_Request requests[8];
    MPI_Irecv(&U(1, 0),             1, row,    bottom, 0, comm, &requests[0]);
    MPI_Irecv(&U(1, jmaxLocal + 1), 1, row,    top,    1, comm, &requests[1]);
    MPI_Irecv(&U(0, 1),             1, column, left,   2, comm, &requests[2]);
    MPI_Irecv(&U(imaxLocal + 1, 1), 1, column, right,  3, comm, &requests[3]);
    MPI_Isend(&U(1, jmaxLocal),     1, row,    top,    0, comm, &requests[4]);
    MPI_Isend(&U(1, 1),             1, row,    bottom, 1, comm, &requests[5]);
    MPI_Isend(&U(imaxLocal, 1),     1, column, right,  2, comm, &requests[6]);
    MPI_Isend(&U(1, 1),             1, column, left,   3, comm, &requests[7]);
    MPI_Waitall(8, requests, MPI_STATUSES_IGNORE);

    double local_change = 0.0;
    for (int j = 1; j <= jmaxLocal; ++j)
        for (int i = 1; i <= imaxLocal; ++i) {
            UNEW(i, j) = 0.25 * (U(i - 1, j) + U(i + 1, j) + U(i, j - 1) + U(i, j + 1));
            double diff = UNEW(i, j) - U(i, j);
            if (diff < 0.0) diff = -diff;
            if (diff > local_change) local_change = diff;
        }
    double *tmp = u;
    u = unew;
    unew = tmp;
    MPI_Allreduce(&local_change, &change, 1, MPI_DOUBLE, MPI_MAX, comm);
}

double local_err = 0.0, err;
for (int j = 1; j <= jmaxLocal; ++j)
    for (int i = 1; i <= imaxLocal; ++i) {
        double diff = U(i, j) - (X(i) + Y(j));
        if (diff < 0.0) diff = -diff;
        if (diff > local_err) local_err = diff;
    }
MPI_Reduce(&local_err, &err, 1, MPI_DOUBLE, MPI_MAX, 0, comm);
if (rank == 0)
    printf("%d x %d processes x %d threads: %d iterations, last change %.6e, max. error %.6e\n",
           dims[0], dims[1], omp_get_max_threads(), iter, change, err);

MPI_Type_free(&column);
MPI_Type_free(&row);
MPI_Comm_free(&comm);
free(u);
free(unew);

## Summary

* Hybrid MPI+OpenMP programs run fewer MPI processes, each with several threads.
* Initialize MPI with `MPI_Init_thread()` and check the provided level: `FUNNELED` for threads at all, `MULTIPLE` only if several threads call MPI concurrently.
* The **funneled** pattern — threads compute, the main thread communicates — is the most common and most portable one.
* With `MPI_THREAD_MULTIPLE`, keep the messages of different threads apart with tags, and give every thread that calls collectives its own communicator — one communicator per thread also lets current libraries communicate for the threads in parallel.
* **Partitioned communication** (MPI 4.0) lets threads contribute the partitions of one message with `MPI_Pready()`.
* Placement and pinning of processes **and** threads are essential, and non-portable — check where everything runs.
* Initialize data in parallel (first touch), or use one process per ccNUMA domain.
* OpenMP loop schedules can fix load imbalance cheaply.
* Go hybrid only for a concrete reason: memory, load balance, overlap, GPUs, or scalability — and check whether pure MPI with current library features (shared-memory windows, progress threads, node-aware collectives) solves it, too.

| Call / construct | Purpose |
|---|---|
| `MPI_Init_thread()` | initialize MPI with a thread support level |
| `MPI_Query_thread()`, `MPI_Is_thread_main()` | query the level / whether the calling thread is the main thread |
| `MPI_Comm_dup()`, `MPI_Comm_dup_with_info()` | a separate communicator, e.g. one per thread, optionally with `mpi_assert_*` hints |
| `MPI_Psend_init()`, `MPI_Precv_init()` | set up a partitioned send / receive |
| `MPI_Pready()`, `MPI_Parrived()` | mark a partition ready / check whether it has arrived |
| `#pragma omp parallel` | parallel region: fork a team of threads |
| `#pragma omp for schedule(…) reduction(…)` | distribute loop iterations among the threads |
| `#pragma omp single`, `masked` (formerly `master`), `barrier` | execute by one thread / by the main thread / wait for all |
| `omp_get_thread_num()`, `omp_get_num_threads()`, `omp_get_wtime()` | thread id, team size, wall-clock time |
| `OMP_NUM_THREADS`, `OMP_PLACES`, `OMP_PROC_BIND` | number of threads and their pinning |

[Back to the overview](../overview.ipynb)